# Packet P-036 — Synthetic Noise Augmentation Test

**Decision question:** Does augmenting training data with noisy copies of long-lived devices (T80 > 1000h) improve model robustness or reduce P-028's systematic underprediction?

**Method:** Generate synthetic training examples by adding Gaussian noise to long-lived devices at 2x and 5x ratios. Train ET, evaluate tau-b and residual bias in the 1000h+ range.

**Fastest falsifier:** 5x augmentation, single split. If tau-b doesn't improve and 1000h+ bias doesn't shrink, augmentation is not useful.

**Success:** Augmented model reduces 1000h+ mean signed error by ≥30% while maintaining tau-b within 0.01.
**Kill:** Augmented model degrades tau-b by > 0.02, or 1000h+ bias doesn't improve.

In [1]:
"""Cell 1 — Load data, identify long-lived devices."""
import pandas as pd
import numpy as np
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import train_test_split
from scipy.stats import kendalltau
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('perovskite_stability_clean.csv')
FEATURES = [
    'Perovskite_band_gap', 'Pb', 'Sn', 'I', 'Br', 'Cl',
    'MA', 'FA', 'Cs',
    'first_Prvskt_annealing_temperature', 'first_Prvskt_thermal_annealing_time',
    'Perovskite_thickness', 'Cell_area_measured',
    'JV_default_Voc', 'JV_default_Jsc', 'JV_default_FF'
]
TARGET = 'Stability_PCE_T80'
ET_PARAMS = dict(n_estimators=700, max_features='sqrt', min_samples_split=3,
                 min_samples_leaf=1, bootstrap=False, random_state=42)

X = df[FEATURES].fillna(0)
y = np.log1p(df[TARGET])

# Identify long-lived devices (T80 > 1000h)
long_lived_mask = df[TARGET] > 1000
n_long = long_lived_mask.sum()
print(f"Dataset: {len(df)} devices")
print(f"Long-lived (T80 > 1000h): {n_long} ({n_long/len(df):.1%})")
print(f"T80 distribution:")
print(f"  Mean: {df[TARGET].mean():.0f}h")
print(f"  Median: {df[TARGET].median():.0f}h")
print(f"  P90: {df[TARGET].quantile(0.9):.0f}h")
print(f"  Max: {df[TARGET].max():.0f}h")

Dataset: 1543 devices
Long-lived (T80 > 1000h): 109 (7.1%)
T80 distribution:
  Mean: 374h
  Median: 150h
  P90: 910h
  Max: 8400h


In [2]:
"""Cell 2 — Augmentation experiment across 10 splits."""

N_SPLITS = 10
NOISE_SIGMA = 0.05  # 5% of feature std
AUGMENT_RATIOS = [1, 2, 5]  # 1x = no augment, 2x, 5x

results = {r: {'taus': [], 'bias_1000': []} for r in AUGMENT_RATIOS}

for seed in range(N_SPLITS):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=seed)
    
    # Identify long-lived devices in training set
    tr_idx = X_tr.index
    ll_mask = df.loc[tr_idx, TARGET] > 1000
    X_ll = X_tr[ll_mask]
    y_ll = y_tr[ll_mask]
    
    # Test set long-lived mask for bias calculation
    te_idx = X_te.index
    te_ll_mask = df.loc[te_idx, TARGET] > 1000
    
    for ratio in AUGMENT_RATIOS:
        if ratio == 1:
            # Baseline: no augmentation
            X_aug = X_tr.copy()
            y_aug = y_tr.copy()
        else:
            # Generate synthetic copies of long-lived devices
            rng = np.random.RandomState(seed + ratio * 100)
            feat_stds = X_tr.std().values
            
            synth_X_list = [X_tr]
            synth_y_list = [y_tr]
            
            for _ in range(ratio - 1):
                noise = rng.normal(0, NOISE_SIGMA, size=X_ll.shape) * feat_stds
                X_noisy = X_ll + noise
                synth_X_list.append(X_noisy)
                synth_y_list.append(y_ll)
            
            X_aug = pd.concat(synth_X_list, ignore_index=True)
            y_aug = pd.concat(synth_y_list, ignore_index=True)
        
        # Train
        params = ET_PARAMS.copy()
        params['random_state'] = seed
        model = ExtraTreesRegressor(**params)
        model.fit(X_aug, y_aug)
        preds = model.predict(X_te)
        
        # Overall tau-b
        tau, _ = kendalltau(y_te, preds)
        results[ratio]['taus'].append(tau)
        
        # Bias on 1000h+ devices (mean signed error in log space)
        if te_ll_mask.sum() > 0:
            ll_preds = preds[te_ll_mask.values]
            ll_actual = y_te[te_ll_mask].values
            bias = np.mean(ll_preds - ll_actual)  # negative = underprediction
            results[ratio]['bias_1000'].append(bias)
    
    if (seed + 1) % 5 == 0:
        print(f"  Completed {seed+1}/{N_SPLITS} splits")

# Report
print(f"\n{'=' * 70}")
print("AUGMENTATION RESULTS")
print(f"{'=' * 70}")
print(f"{'Ratio':<8} {'Mean tau-b':>11} {'Std':>8} {'1000h+ bias':>13} {'Aug devices':>12}")
print("-" * 55)

for ratio in AUGMENT_RATIOS:
    r = results[ratio]
    mean_tau = np.mean(r['taus'])
    std_tau = np.std(r['taus'])
    mean_bias = np.mean(r['bias_1000']) if r['bias_1000'] else float('nan')
    n_ll = ll_mask.sum()
    aug_n = n_ll * (ratio - 1) if ratio > 1 else 0
    print(f"{ratio}x{'':<5} {mean_tau:>11.4f} {std_tau:>8.4f} {mean_bias:>+13.4f} {aug_n:>12}")

baseline_tau = np.mean(results[1]['taus'])
baseline_bias = np.mean(results[1]['bias_1000'])
best_ratio = max([2, 5], key=lambda r: -abs(np.mean(results[r]['bias_1000'])))
best_tau = np.mean(results[best_ratio]['taus'])
best_bias = np.mean(results[best_ratio]['bias_1000'])

  Completed 5/10 splits


  Completed 10/10 splits

AUGMENTATION RESULTS
Ratio     Mean tau-b      Std   1000h+ bias  Aug devices
-------------------------------------------------------
1x           0.2987   0.0437       -1.9220            0
2x           0.2998   0.0422       -1.9042           82
5x           0.3029   0.0430       -1.8631          328


In [3]:
"""Cell 3 — Evaluate and save."""

# Success: reduces 1000h+ bias by >=30% while maintaining tau-b within 0.01
# Kill: degrades tau-b by >0.02 or bias doesn't improve

tau_delta_best = best_tau - baseline_tau
bias_reduction = 1 - abs(best_bias) / abs(baseline_bias) if abs(baseline_bias) > 1e-6 else 0

print("── Augmentation Evaluation ──\n")
print(f"Baseline tau-b:     {baseline_tau:.4f}")
print(f"Best augmented:     {best_tau:.4f} ({best_ratio}x)")
print(f"Tau-b delta:        {tau_delta_best:+.4f}")
print(f"Baseline 1000h+ bias: {baseline_bias:+.4f}")
print(f"Best 1000h+ bias:     {best_bias:+.4f}")
print(f"Bias reduction:       {bias_reduction:.1%}")

tau_ok = abs(tau_delta_best) <= 0.01
bias_improved = bias_reduction >= 0.30
tau_degraded = tau_delta_best < -0.02

if tau_degraded:
    status = "Negative"
elif tau_ok and bias_improved:
    status = "Confirmed"
elif bias_improved or (not tau_degraded and bias_reduction > 0):
    status = "Promising"
else:
    status = "Negative"

# Save
save_rows = []
for ratio in AUGMENT_RATIOS:
    r = results[ratio]
    save_rows.append({
        'ratio': f'{ratio}x',
        'mean_tau': np.mean(r['taus']),
        'std_tau': np.std(r['taus']),
        'mean_bias_1000h': np.mean(r['bias_1000']) if r['bias_1000'] else float('nan')
    })
pd.DataFrame(save_rows).to_csv('Packet_P036_Synthetic_Augmentation.csv', index=False)
print(f"\nSaved: Packet_P036_Synthetic_Augmentation.csv")

print(f"\n{'=' * 60}")
print(f"P-036 STATUS: {status}")
print(f"{'=' * 60}")
if status == "Confirmed":
    print("Augmentation reduces long-lived underprediction without hurting ranking.")
elif status == "Promising":
    print("Some improvement but not meeting full success criteria.")
else:
    print("Synthetic augmentation does not help. Need genuinely new data (per P-012).")

── Augmentation Evaluation ──

Baseline tau-b:     0.2987
Best augmented:     0.3029 (5x)
Tau-b delta:        +0.0042
Baseline 1000h+ bias: -1.9220
Best 1000h+ bias:     -1.8631
Bias reduction:       3.1%

Saved: Packet_P036_Synthetic_Augmentation.csv

P-036 STATUS: Promising
Some improvement but not meeting full success criteria.
